# PolyWhisper Training — English + Hindi
Resumable training on Colab Free (T4). Run all cells. If session breaks, re-run all cells to resume.

In [ ]:
# CELL 1: Install
!pip install -q transformers==4.44.2 datasets==3.1.0 soundfile librosa

import torch, torch.nn as nn, numpy as np, json, os, time, math
from pathlib import Path
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
from google.colab import drive

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# CELL 2: Config + Model

WHISPER_MODEL = "openai/whisper-base"
LANGUAGES = ["en", "hi"]
NUM_EPOCHS = 10
BATCH_SIZE = 16
LR = 1e-3
MAX_AUDIO_SEC = 15.0
MAX_LABEL_LEN = 128
AD_HID = 256
AD_LAYERS = 2
AD_HEADS = 4
AD_FFN = 1024
AD_RANK = 8

drive.mount("/content/drive")
SAVE_DIR = Path("/content/drive/MyDrive/polywhisper")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = SAVE_DIR / "adapters"
ADAPTER_DIR.mkdir(exist_ok=True)
DATA_DIR = SAVE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
STATE_FILE = SAVE_DIR / "training_state.json"

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL)
ENCODE_DIM = 512
VOCAB_SIZE = len(processor.tokenizer)  # 51865, covers all special tokens

print(f"Vocab: {VOCAB_SIZE}, Encoder dim: {ENCODE_DIM}")

class LowRank(nn.Module):
    def __init__(self, i, o, r):
        super().__init__()
        self.a = nn.Linear(i, r, bias=False)
        self.b = nn.Linear(r, o, bias=False)
        nn.init.zeros_(self.b.weight)
    def forward(self, x):
        return self.b(self.a(x))

class SelfAttn(nn.Module):
    def __init__(self, d, h, r):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)
        self.o = nn.Linear(d, d)
        self.qa = LowRank(d, d, r)
        self.va = LowRank(d, d, r)
        self.h, self.dh = h, d // h
        self.sc = math.sqrt(self.dh)
        self.drop = nn.Dropout(0.1)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x) + self.qa(x)
        k, v = self.k(x), self.v(x) + self.va(x)
        def rs(t): return t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = rs(q), rs(k), rs(v)
        a = self.drop(torch.softmax((q @ k.transpose(-2, -1)) / self.sc, dim=-1))
        return self.o((a @ v).transpose(1, 2).contiguous().view(B, T, -1))

class CrossAttn(nn.Module):
    def __init__(self, d, ed, h, r):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(ed, d)
        self.v = nn.Linear(ed, d)
        self.o = nn.Linear(d, d)
        self.qa = LowRank(d, d, r)
        self.va = LowRank(ed, d, r)
        self.h, self.dh = h, d // h
        self.sc = math.sqrt(self.dh)
        self.drop = nn.Dropout(0.1)
    def forward(self, x, enc):
        B, Td, _ = x.shape
        Te = enc.size(1)
        q = self.q(x) + self.qa(x)
        k, v = self.k(enc), self.v(enc) + self.va(enc)
        def rs(t, T): return t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = rs(q, Td), rs(k, Te), rs(v, Te)
        a = self.drop(torch.softmax((q @ k.transpose(-2, -1)) / self.sc, dim=-1))
        return self.o((a @ v).transpose(1, 2).contiguous().view(B, Td, -1))

class AdaptLayer(nn.Module):
    def __init__(self, d, ed, h, ffn, r):
        super().__init__()
        self.sa = SelfAttn(d, h, r)
        self.ca = CrossAttn(d, ed, h, r)
        self.ff = nn.Sequential(nn.Linear(d, ffn), nn.GELU(), nn.Dropout(0.1), nn.Linear(ffn, d))
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)
        self.n3 = nn.LayerNorm(d)
        self.drop = nn.Dropout(0.1)
    def forward(self, x, enc):
        x = x + self.drop(self.sa(self.n1(x)))
        x = x + self.drop(self.ca(self.n2(x), enc))
        x = x + self.drop(self.ff(self.n3(x)))
        return x

class LangAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.te = nn.Embedding(VOCAB_SIZE, AD_HID)
        self.pe = nn.Embedding(MAX_LABEL_LEN, AD_HID)
        self.layers = nn.ModuleList([AdaptLayer(AD_HID, ENCODE_DIM, AD_HEADS, AD_FFN, AD_RANK) for _ in range(AD_LAYERS)])
        self.out = nn.Linear(AD_HID, VOCAB_SIZE)
        self.norm = nn.LayerNorm(AD_HID)
        self.drop = nn.Dropout(0.1)
    def forward(self, enc, ids):
        ids = ids.clamp(min=0, max=VOCAB_SIZE - 1)
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        x = self.drop(self.te(ids) + self.pe(pos))
        for l in self.layers:
            x = l(x, enc)
        return self.out(self.norm(x))

class PolyWhisper(nn.Module):
    def __init__(self):
        super().__init__()
        self.whisper = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL)
        for p in self.whisper.model.encoder.parameters():
            p.requires_grad = False
        self.adapters = nn.ModuleDict({l: LangAdapter() for l in LANGUAGES})
    def forward(self, feat, dec_ids, lang):
        enc = self.whisper.model.encoder(feat).last_hidden_state
        return self.adapters[lang](enc, dec_ids)
    def save(self, path):
        torch.save({n: a.state_dict() for n, a in self.adapters.items()}, path)
    def load_ckpt(self, path, device="cpu"):
        st = torch.load(path, map_location=device, weights_only=True)
        for n, s in st.items():
            if n in self.adapters:
                self.adapters[n].load_state_dict(s)

model = PolyWhisper().to("cuda")
tp = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {tp/1e6:.1f}M")

In [ ]:
# CELL 3: Data

import soundfile as sf

class Fleurs:
    def __init__(self, lang, split):
        self.cache = DATA_DIR / f"fleurs_{lang}_{split}.json"
        self.adir = DATA_DIR / f"audio_{lang}_{split}"
        self.adir.mkdir(parents=True, exist_ok=True)
        self.data = self._load(lang, split)
    def _load(self, lang, split):
        cfg = {"en": "en_us", "hi": "hi_in"}[lang]
        if self.cache.exists():
            print(f"  Cached {lang}/{split}")
            return json.load(open(self.cache))
        print(f"  Downloading FLEURS {lang}/{split}...")
        ds = load_dataset("google/fleurs", cfg, split=split)
        recs = []
        for i, item in enumerate(tqdm(ds, desc=f"  {lang}/{split}")):
            wav = self.adir / f"{i:05d}.wav"
            a = np.array(item["audio"]["array"], dtype=np.float32)
            a = a[:int(MAX_AUDIO_SEC * 16000)]
            sf.write(str(wav), a, 16000)
            t = item["transcription"]
            recs.append({"wav": str(wav), "text": t.lower().strip() if lang == "en" else t.strip()})
        json.dump(recs, open(self.cache, "w"))
        return recs
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        r = self.data[i]
        a, _ = sf.read(r["wav"])
        return {"audio": a.astype(np.float32), "text": r["text"]}

def collate(batch):
    auds = [b["audio"] for b in batch]
    txts = [b["text"] for b in batch]
    ai = processor.feature_extractor(auds, sampling_rate=16000, return_tensors="pt",
        padding=True, truncation=True, max_length=int(30 * 16000))
    f = ai["input_features"]
    if f.shape[-1] < 3000:
        f = torch.nn.functional.pad(f, (0, 3000 - f.shape[-1]))
    li = processor.tokenizer(txts, return_tensors="pt", padding="max_length",
        max_length=MAX_LABEL_LEN, truncation=True)
    li["input_ids"] = li["input_ids"].masked_fill(li["attention_mask"] == 0, -100)
    return {"input_features": f, "labels": li["input_ids"]}

train_loaders = {}
val_loaders = {}
for lang in LANGUAGES:
    tr, va = Fleurs(lang, "train"), Fleurs(lang, "validation")
    train_loaders[lang] = torch.utils.data.DataLoader(tr, batch_size=BATCH_SIZE,
        shuffle=True, collate_fn=collate, num_workers=2, pin_memory=True)
    val_loaders[lang] = torch.utils.data.DataLoader(va, batch_size=BATCH_SIZE,
        shuffle=False, collate_fn=collate, num_workers=2, pin_memory=True)
    print(f"  {lang}: {len(tr)} train, {len(va)} val")
print("Data ready")

In [ ]:
# CELL 4: Training (resumable)

def load_state():
    if STATE_FILE.exists():
        s = json.load(open(STATE_FILE))
        print(f"Resumed: {s.get('done',{})}")
        return s
    return {"done": {}, "step": 0, "log": {}}

def save_state(s):
    json.dump(s, open(STATE_FILE, "w"))

def get_ckpt(lang):
    best = ADAPTER_DIR / f"{lang}_best.pt"
    if best.exists():
        return best
    eps = sorted(ADAPTER_DIR.glob(f"{lang}_ep*.pt"))
    return eps[-1] if eps else None

state = load_state()
start_epoch = max((state["done"].get(l, 0) for l in LANGUAGES), default=0)
print(f"Starting from epoch {start_epoch + 1}")

for lang in LANGUAGES:
    p = get_ckpt(lang)
    if p:
        model.load_ckpt(p)
        print(f"  Loaded {lang}: {p.name}")

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

@torch.no_grad()
def evaluate(lang):
    model.eval()
    total, n = 0, 0
    for b in val_loaders[lang]:
        feat = b["input_features"].to("cuda")
        lab = b["labels"].to("cuda")
        dec_in = lab[:, :-1].clamp(min=0)
        out = model(feat, dec_in, lang)
        total += criterion(out.reshape(-1, out.size(-1)), lab[:, 1:].reshape(-1)).item()
        n += 1
    model.train()
    return total / max(n, 1)

print("=" * 50)
print("TRAINING START")
print("=" * 50)

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    epoch_log = {}
    for lang in LANGUAGES:
        if state["done"].get(lang, 0) > epoch:
            print(f"  [{lang}] Done, skip")
            continue
        model.train()
        ep_loss, n = 0, 0
        for b in tqdm(train_loaders[lang], desc=f"  {lang}"):
            feat = b["input_features"].to("cuda")
            lab = b["labels"].to("cuda")
            dec_in = lab[:, :-1].clamp(min=0)
            out = model(feat, dec_in, lang)
            loss = criterion(out.reshape(-1, out.size(-1)), lab[:, 1:].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            ep_loss += loss.item()
            n += 1
            state["step"] += 1
        avg = ep_loss / max(n, 1)
        val = evaluate(lang)
        epoch_log[lang] = {"train": avg, "val": val}
        ckpt = ADAPTER_DIR / f"{lang}_ep{epoch + 1}.pt"
        model.save(str(ckpt))
        state["done"][lang] = epoch + 1
        save_state(state)
        print(f"  [{lang}] train={avg:.4f} val={val:.4f} saved={ckpt.name}")
    state["log"][f"ep{epoch + 1}"] = epoch_log
    save_state(state)

# Save best
import shutil
for lang in LANGUAGES:
    best_v, best_ep = float("inf"), 0
    for ek, lv in state["log"].items():
        if lang in lv and lv[lang]["val"] < best_v:
            best_v, best_ep = lv[lang]["val"], int(ek[2:])
    src = ADAPTER_DIR / f"{lang}_ep{best_ep}.pt"
    if src.exists():
        shutil.copy2(src, ADAPTER_DIR / f"{lang}_best.pt")
        print(f"Best {lang}: epoch {best_ep} (val={best_v:.4f})")

state["done"]["__all__"] = NUM_EPOCHS
save_state(state)
print("\nTRAINING COMPLETE!")

In [ ]:
# CELL 5: Download
from google.colab import files
for lang in LANGUAGES:
    p = ADAPTER_DIR / f"{lang}_best.pt"
    if p.exists():
        files.download(str(p))
        print(f"Downloaded {lang}_best.pt")
print(f"Also at: {ADAPTER_DIR}")